# 사용자 행동 인식 데이터 (다중분류)

https://www.kaggle.com/datasets/uciml/human-activity-recognition-with-smartphones 

## 요구사항
- 이 데이터에 적합한 최고의 성능을 가진 분류 모델을 찾아보세요.
- 최적의 하이퍼 파라미터설정이 필요합니다.

아래는 모델별 성능비교표 예시입니다.

모델명 | Accuracy | Precision | Recall | F1-score | AUC | 기타 메모
-- | -- | -- | -- | -- | -- | --
Logistic Regression | 0.89 | 0.88 | 0.87 | 0.87 | 0.91 | 베이스라인 모델
Random Forest | 0.92 | 0.91 | 0.90 | 0.91 | 0.95 | 파라미터 기본값
XGBoost | 0.94 | 0.93 | 0.92 | 0.93 | 0.96 | EarlyStopping 적용
SVM (RBF) | 0.90 | 0.89 | 0.88 | 0.88 | 0.93 | 표준화 필수
KNN (k=5) | 0.85 | 0.84 | 0.83 | 0.83 | 0.87 | 거리기반 특성 영향


In [2]:
!pip install xgboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 32.9 MB/s  0:00:00


In [6]:
# =========================================
# 1. 라이브러리
# =========================================
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

In [7]:
# =========================================
# 2. 데이터 불러오기
# =========================================
train_df = pd.read_csv("../data/human_train.csv")
test_df = pd.read_csv("../data/human_test.csv")

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)

# target / leakage 컬럼 처리
TARGET_COL = "Activity"
LEAKAGE_COL = "subject"   # 사람 ID라서 보통 제외 권장

X_train = train_df.drop(columns=[TARGET_COL, LEAKAGE_COL])
X_test = test_df.drop(columns=[TARGET_COL, LEAKAGE_COL])

y_train_raw = train_df[TARGET_COL]
y_test_raw = test_df[TARGET_COL]

# 라벨 인코딩
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test = label_encoder.transform(y_test_raw)

class_names = list(label_encoder.classes_)
num_classes = len(class_names)

print("classes:", class_names)

train shape: (7352, 563)
test shape : (2947, 563)
classes: ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']


In [8]:
# =========================================
# 3. LightGBM용 컬럼명 정리
#    (특수문자 때문에 에러가 날 수 있음)
# =========================================
def sanitize_columns(columns):
    new_cols = []
    for c in columns:
        c2 = re.sub(r"[^A-Za-z0-9_]+", "_", c)
        new_cols.append(c2)
    return new_cols

X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()

safe_cols = sanitize_columns(X_train.columns)
X_train_lgb.columns = safe_cols
X_test_lgb.columns = safe_cols

In [9]:
# =========================================
# 4. 평가 함수
# =========================================
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name="model"):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_te, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_te, y_pred, average="weighted", zero_division=0)

    auc = np.nan
    try:
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_te)
            y_te_bin = label_binarize(y_te, classes=np.arange(num_classes))
            auc = roc_auc_score(
                y_te_bin,
                y_proba,
                average="weighted",
                multi_class="ovr"
            )
    except Exception:
        pass

    return {
        "모델명": model_name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-score": round(f1, 4),
        "AUC": round(auc, 4) if not np.isnan(auc) else None
    }

In [10]:
# =========================================
# 5. 베이스라인 모델 비교
# =========================================
baseline_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42))
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(C=1.0, random_state=42))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ),
    "LightGBM": LGBMClassifier(
        objective="multiclass",
        num_class=num_classes,
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )
}

baseline_results = []

for name, model in baseline_models.items():
    print(f"Evaluating: {name}")
    if name == "LightGBM":
        result = evaluate_model(model, X_train_lgb, y_train, X_test_lgb, y_test, name)
    else:
        result = evaluate_model(model, X_train, y_train, X_test, y_test, name)
    baseline_results.append(result)

baseline_df = pd.DataFrame(baseline_results).sort_values(by="Accuracy", ascending=False)
print("\n=== 베이스라인 성능 비교 ===")
print(baseline_df)

Evaluating: Logistic Regression
Evaluating: Random Forest
Evaluating: SVM (RBF)
Evaluating: Linear SVM
Evaluating: KNN
Evaluating: XGBoost
Evaluating: LightGBM

=== 베이스라인 성능 비교 ===
                   모델명  Accuracy  Precision  Recall  F1-score     AUC
3           Linear SVM    0.9627     0.9637  0.9627    0.9625     NaN
0  Logistic Regression    0.9549     0.9567  0.9549    0.9548  0.9975
2            SVM (RBF)    0.9542     0.9549  0.9542    0.9540  0.9977
5              XGBoost    0.9348     0.9364  0.9348    0.9346  0.9970
6             LightGBM    0.9348     0.9363  0.9348    0.9346  0.9967
1        Random Forest    0.9281     0.9293  0.9281    0.9278  0.9956
4                  KNN    0.8836     0.8906  0.8836    0.8827  0.9764


In [11]:
# =========================================
# 6. 상위 후보 모델 튜닝
#    추천: XGBoost, LightGBM, SVM(RBF)
# =========================================
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# 6-1. XGBoost 튜닝
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_param_dist = {
    "n_estimators": [200, 300, 400, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [4, 5, 6, 8],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("\nTuning XGBoost...")
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_
print("Best XGBoost params:", xgb_search.best_params_)
print("Best XGBoost CV score:", xgb_search.best_score_)


# 6-2. LightGBM 튜닝
lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=num_classes,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm_param_dist = {
    "n_estimators": [200, 300, 400, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 5, 10, 15],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_samples": [10, 20, 30]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("\nTuning LightGBM...")
lgbm_search.fit(X_train_lgb, y_train)
best_lgbm = lgbm_search.best_estimator_
print("Best LightGBM params:", lgbm_search.best_params_)
print("Best LightGBM CV score:", lgbm_search.best_score_)


# 6-3. SVM (RBF) 튜닝
svm_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", probability=True, random_state=42))
])

svm_param_dist = {
    "clf__C": [0.1, 1, 10, 30, 50, 100],
    "clf__gamma": ["scale", 0.001, 0.01, 0.1, 1]
}

svm_search = RandomizedSearchCV(
    estimator=svm_rbf,
    param_distributions=svm_param_dist,
    n_iter=12,
    scoring="accuracy",
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("\nTuning SVM (RBF)...")
svm_search.fit(X_train, y_train)
best_svm = svm_search.best_estimator_
print("Best SVM params:", svm_search.best_params_)
print("Best SVM CV score:", svm_search.best_score_)


Tuning XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best XGBoost params: {'subsample': 0.9, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}
Best XGBoost CV score: 0.989390586099801

Tuning LightGBM...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best LightGBM params: {'subsample': 1.0, 'num_leaves': 63, 'n_estimators': 500, 'min_child_samples': 30, 'max_depth': 5, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
Best LightGBM CV score: 0.9937429953621596

Tuning SVM (RBF)...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best SVM params: {'clf__gamma': 0.001, 'clf__C': 30}
Best SVM CV score: 0.986125946094472


In [12]:
# =========================================
# 7. 튜닝 후 최종 성능 비교
# =========================================
final_models = {
    "Best XGBoost": (best_xgb, X_train, X_test),
    "Best LightGBM": (best_lgbm, X_train_lgb, X_test_lgb),
    "Best SVM (RBF)": (best_svm, X_train, X_test)
}

final_results = []

for name, (model, X_tr, X_te) in final_models.items():
    print(f"\nFinal evaluation: {name}")
    result = evaluate_model(model, X_tr, y_train, X_te, y_test, name)
    final_results.append(result)

final_df = pd.DataFrame(final_results).sort_values(by="Accuracy", ascending=False)

print("\n=== 최종 튜닝 모델 성능 비교 ===")
print(final_df)


Final evaluation: Best XGBoost

Final evaluation: Best LightGBM

Final evaluation: Best SVM (RBF)

=== 최종 튜닝 모델 성능 비교 ===
              모델명  Accuracy  Precision  Recall  F1-score     AUC
2  Best SVM (RBF)    0.9566     0.9574  0.9566    0.9564  0.9977
1   Best LightGBM    0.9427     0.9437  0.9427    0.9425  0.9975
0    Best XGBoost    0.9413     0.9425  0.9413    0.9411  0.9974


In [13]:
# =========================================
# 8. 최종 우승 모델 상세 리포트
# =========================================
best_row = final_df.iloc[0]
best_model_name = best_row["모델명"]

if best_model_name == "Best XGBoost":
    winner_model = best_xgb
    winner_X_test = X_test
    winner_X_train = X_train
elif best_model_name == "Best LightGBM":
    winner_model = best_lgbm
    winner_X_test = X_test_lgb
    winner_X_train = X_train_lgb
else:
    winner_model = best_svm
    winner_X_test = X_test
    winner_X_train = X_train

winner_model.fit(winner_X_train, y_train)
winner_pred = winner_model.predict(winner_X_test)

print("\n=== 최종 우승 모델 ===")
print(best_model_name)

print("\n=== 분류 리포트 ===")
print(classification_report(y_test, winner_pred, target_names=class_names, zero_division=0))


=== 최종 우승 모델 ===
Best SVM (RBF)

=== 분류 리포트 ===
                    precision    recall  f1-score   support

            LAYING       0.99      1.00      1.00       537
           SITTING       0.96      0.89      0.93       491
          STANDING       0.91      0.97      0.94       532
           WALKING       0.96      0.98      0.97       496
WALKING_DOWNSTAIRS       0.98      0.93      0.95       420
  WALKING_UPSTAIRS       0.93      0.96      0.95       471

          accuracy                           0.96      2947
         macro avg       0.96      0.96      0.96      2947
      weighted avg       0.96      0.96      0.96      2947

